# Time Series with Deep Learning
Time series data is pervasive: stock prices, sensor readings, weather, IoT, ECG signals. Deep learning has enabled major advances in time series forecasting and classification. This notebook covers LSTM-based models, Temporal Convolutional Networks (TCN), and Transformer-based approaches (Informer).

## 1. Time Series Fundamentals
Key characteristics that differ from other data types:
- **Temporal ordering**: past predicts future; future cannot influence past (no data leakage)
- **Stationarity**: statistical properties (mean, variance) ideally constant over time
- **Seasonality**: regular periodic patterns (daily, weekly, yearly)
- **Trend**: long-range directional movement

**Preprocessing steps:**
- Differencing to remove trends
- Log transform to stabilize variance
- MinMax or Standard scaling (applied using only training set statistics)
- Sliding window: convert series into (input_window, forecast_horizon) pairs

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# Create sliding window dataset
def create_sliding_window(series, input_len, horizon):
    X, y = [], []
    for i in range(len(series) - input_len - horizon + 1):
        X.append(series[i : i + input_len])
        y.append(series[i + input_len : i + input_len + horizon])
    return np.array(X), np.array(y)

np.random.seed(42)
t = np.linspace(0, 4*np.pi, 1000)
series = np.sin(t) + 0.1 * np.random.randn(1000)  # Noisy sinusoid

X, y = create_sliding_window(series, input_len=60, horizon=10)
print("Input windows shape:", X.shape)   # (N, 60)
print("Forecast targets shape:", y.shape)  # (N, 10)

Input windows shape: (931, 60)
Forecast targets shape: (931, 10)


## 2. LSTM for Time Series
Stacked LSTMs are the classic approach for sequence forecasting.

**Input shape**: (batch, timesteps, features)
**Output**: Point forecast or distribution over forecast horizon

In [2]:
# LSTM Forecasting Model
lstm_model = models.Sequential([
    layers.Input(shape=(60, 1)),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(64),
    layers.Dense(32, activation='relu'),
    layers.Dense(10)           # 10-step forecast horizon
])
lstm_model.compile(optimizer='adam', loss='huber')
lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━┳━━━━━━━┳━━━━┓
┃ Layer   ┃ Outp… ┃ P… ┃
┃ (type)  ┃ Shape ┃  # ┃
┡━━━━━━━━━╇━━━━━━━╇━━━━┩
│ lstm    │ (Non… │ 6… │
│ (LSTM)  │ 60,   │    │
│         │ 128)  │    │
├─────────┼───────┼────┤
│ dropout │ (Non… │  0 │
│ (Dropo… │ 60,   │    │
│         │ 128)  │    │
├─────────┼───────┼────┤
│ lstm_1  │ (Non… │ 4… │
│ (LSTM)  │ 64)   │    │
├─────────┼───────┼────┤
│ dense   │ (Non… │ 2… │
│ (Dense) │ 32)   │    │
├─────────┼───────┼────┤
│ dense_1 │ (Non… │ 3… │
│ (Dense) │ 10)   │    │
└─────────┴───────┴────┘

 Total params: 118,378 (462.41 KB)

 Trainable params: 118,378 (462.41 KB)

 Non-trainable params: 0 (0.00 B)

## 3. Temporal Convolutional Network (TCN)
A 1D CNN-based architecture designed specifically for sequential data. Key properties:
- **Dilated Causal Convolutions**: ensures the model only uses past information (causality)
- **Residual Connections**: stabilize gradients in deep networks
- **Dilation doubling**: [1, 2, 4, 8, 16, ...] exponentially increases the receptive field
- **Advantages over LSTM**: fully parallelizable, stable gradients, often faster training
- Receptive field size: (kernel_size - 1) * sum(dilations) + 1

In [3]:
def tcn_block(x, filters, kernel_size, dilation_rate):
    # Dilated causal convolution
    conv = layers.Conv1D(filters, kernel_size,
                         dilation_rate=dilation_rate,
                         padding='causal', activation='relu')(x)
    conv = layers.Conv1D(filters, kernel_size,
                         dilation_rate=dilation_rate,
                         padding='causal', activation='relu')(conv)
    # Residual connection (1x1 conv to match dims if needed)
    if x.shape[-1] != filters:
        x = layers.Conv1D(filters, 1)(x)
    return layers.Add()([x, conv])

inp = layers.Input(shape=(60, 1))
x = inp
for dilation in [1, 2, 4, 8, 16]:
    x = tcn_block(x, filters=64, kernel_size=3, dilation_rate=dilation)

x = layers.GlobalAveragePooling1D()(x)
out = layers.Dense(10)(x)   # Forecast horizon

tcn_model = models.Model(inp, out, name="TCN_Forecaster")
tcn_model.compile(optimizer='adam', loss='huber')
tcn_model.summary()

Model: "TCN_Forecaster"

┏━━━━━━┳━━━━━┳━━┳━━━━━━┓
┃ Lay… ┃ Ou… ┃  ┃ Con… ┃
┃ (ty… ┃ Sh… ┃  ┃ to   ┃
┡━━━━━━╇━━━━━╇━━╇━━━━━━┩
│ inp… │ (N… │  │ -    │
│ (In… │ 60, │  │      │
│      │ 1)  │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ inp… │
│ (Co… │ 60, │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ inp… │
│ (Co… │ 60, │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 60, │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ add  │ (N… │  │ con… │
│ (Ad… │ 60, │  │ con… │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ add… │
│ (Co… │ 60, │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 60, │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ add… │ (N… │  │ add… │
│ (Ad… │ 60, │  │ con… │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ add… │
│ (Co… │ 60, │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 60, │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ add… │ (N… │  │ add… │
│ (Ad… │ 60, │  │ con… │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ add… │
│ (Co… │ 60, │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 60, │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ add… │ (N… │  │ add… │
│ (Ad… │ 60, │  │ con… │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ add… │
│ (Co… │ 60, │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ con… │ (N… │  │ con… │
│ (Co… │ 60, │  │      │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ add… │ (N… │  │ add… │
│ (Ad… │ 60, │  │ con… │
│      │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ glo… │ (N… │  │ add… │
│ (Gl… │ 64) │  │      │
├──────┼─────┼──┼──────┤
│ den… │ (N… │  │ glo… │
│ (De… │ 10) │  │      │
└──────┴─────┴──┴──────┘

 Total params: 112,202 (438.29 KB)

 Trainable params: 112,202 (438.29 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Transformer for Time Series (Informer)
Standard Transformers have O(n^2) attention — impractical for long time series (thousands of steps).

**Informer (2021)** introduces several innovations:
- **ProbSparse Attention**: selects only the top-u queries (O(n log n) complexity)
- **Distillation**: halves the temporal dimension between encoder layers to focus on the most important timesteps
- **Generative decoder**: predicts all future steps in one shot (not step-by-step)

Other modern time series Transformers:
- **PatchTST**: divides time series into patches (like ViT for images) — very strong forecasting performance
- **Autoformer**: uses Auto-Correlation mechanism
- **TimesNet**: projects 1D time series to 2D to use CNN-based backbone

# Conclusions and Key Takeaways
- **LSTMs** are the classic baseline but suffer from sequential computation and short-term memory issues on very long series.
- **TCNs** offer equal or better performance with full parallelization, making training much faster.
- **Transformer-based models** (Informer, PatchTST) extend attention to very long horizons efficiently.
- Preprocessing (scaling, differencing, proper windowing without data leakage) is as important as model choice.

# Pros and Cons
**Pros:**
- LSTM: mature, well-understood, good for short sequences; handles irregular sampling with masking
- TCN: parallelizable, stable training, can achieve very large receptive fields with few layers
- Transformer: excels at capturing long-range dependencies; flexible for multi-variate forecasting

**Cons:**
- LSTM: sequential computation prevents GPU parallelism; vanishing gradients for very long horizons
- TCN: fixed receptive field per architecture; must be designed carefully to capture the right horizon
- Transformers: data-hungry; quadratic attention is expensive for very long sequences

# 15 Interview Questions and Answers

1. **What is the sliding window technique for time series?**
   *Answer*: A method to create supervised learning examples from a continuous series by defining an input window of length L and a forecast horizon of H. The window slides one step at a time to create (input, target) pairs.

2. **Why must feature scaling for time series only use training statistics?**
   *Answer*: Using statistics from the full dataset (including test/validation) would constitute data leakage — the model indirectly sees future information during preprocessing. Only fit scalers on training data.

3. **What is the key constraint of causal convolutions?**
   *Answer*: They can only use past and present input values (no future values). This is enforced by left-padding the convolution appropriately, ensuring the t-th output only depends on inputs up to time t.

4. **How does TCN achieve a large receptive field efficiently?**
   *Answer*: By exponentially doubling dilation rates (1, 2, 4, 8, ..., 2^k). With kernel size 2 and k=10 dilations, the receptive field reaches 2^10=1024 timesteps using only ~20 convolution layers.

5. **What is Multivariate vs Univariate Time Series Forecasting?**
   *Answer*: Univariate uses only the single target series to forecast future values. Multivariate uses multiple related time series simultaneously (e.g., temperature, humidity, and wind speed together to forecast energy demand).

6. **What is the problem with applying standard Transformers to long time series?**
   *Answer*: O(L^2) attention complexity is prohibitive for thousands of timesteps. For L=1000 steps, this means 10^6 attention computations per head per layer.

7. **What is ProbSparse Attention in Informer?**
   *Answer*: Only the top-u "dominant queries" (those with high sparseness measure) are used in attention, reducing complexity from O(n^2) to O(n log n) while maintaining most of the predictive power.

8. **What is PatchTST?**
   *Answer*: Divides the time series into fixed-length overlapping patches (similar to ViT image patches) and applies a Transformer with channel-independence. Each "patch" becomes a token; this reduces sequence length and captures local temporal patterns.

9. **How is time series evaluation different from other ML tasks?**
   *Answer*: You must use temporal train/test splits — future data cannot be used for training. Walk-forward validation (time-based k-fold) is used instead of random shuffling.

10. **What metrics are used for time series forecasting?**
    *Answer*: MAE (Mean Absolute Error) for scale-independent assessment; MAPE (Mean Absolute Percentage Error) for relative error; RMSE for penalizing large errors; MASE (Mean Absolute Scaled Error) for comparing against a naive baseline.

11. **What is a Seasonal Decomposition and how does it help deep learning models?**
    *Answer*: Decomposing a series into trend + seasonality + residual. Removing known seasonal patterns (STL decomposition) allows the model to focus on learning the complex residual patterns, often improving performance.

12. **What is the Temporal Fusion Transformer (TFT)?**
    *Answer*: A purpose-built Transformer for multi-horizon time series forecasting. It combines LSTM encoders, Variable Selection Networks, Gated Residual Networks, and multi-head attention to handle known future inputs, static metadata, and observed history jointly.

13. **Why might a simple ARIMA model outperform an LSTM on some datasets?**
    *Answer*: When the series is short (few hundred points), linear, stationary, and well-behaved. Deep models need large datasets to outperform statistical baselines; on small datasets they overfit.

14. **How are anomalies detected in time series using deep learning?**
    *Answer*: Train an LSTM or Transformer on normal windows, then measure reconstruction error on new windows. High reconstruction error indicates anomalous patterns. VAE-based approaches learn a probabilistic model and use log-likelihood as the anomaly score.

15. **What is teacher forcing and why is it used in Seq2Seq forecasting?**
    *Answer*: During training, the ground truth previous output tokens are fed as input to the decoder at each step rather than the model's own predictions. This stabilizes training by providing correct context, though it creates a discrepancy from inference behavior (exposure bias).
